# Brain Tumor Segmentation — Colab Training

Clones the repo, downloads BraTS2020 from Kaggle, mounts Drive for checkpoint persistence, applies memory-safe config overrides, and trains.

**Before running:** Runtime -> Change runtime type -> GPU (T4).

## 1. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/thatavidreader/brain_tumor_segmentation.git
%cd brain_tumor_segmentation
!pip install -q -r requirements.txt

## 2. Upload Kaggle API token
Download `kaggle.json` from https://www.kaggle.com/settings/api (Generate New Token) and upload it below.

In [ ]:
from google.colab import files
files.upload()  # select your downloaded kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 3. Download and restructure the BraTS2020 dataset

In [ ]:
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p data/raw
!unzip -q data/raw/brats20-dataset-training-validation.zip -d data/raw/extracted
!mv data/raw/extracted/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_* data/raw/
!rm -rf data/raw/extracted data/raw/brats20-dataset-training-validation.zip

## 4. Mount Google Drive (for checkpoint persistence across sessions)
Clears any stale/partial mount state first, then mounts fresh.

In [ ]:
!fusermount -uz /content/drive 2>/dev/null
!rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/brain_tumor_seg_checkpoints

## 5. Apply config overrides
`src/train.py` only accepts `--config <path>` (no `--epochs`/`--paths.x` CLI overrides), so we edit `config.yaml` directly:

- `epochs: 100` — shorter run for a first full pass
- `checkpoint_dir` -> Drive, so checkpoints survive a Colab disconnect
- `cache_rate: 0.05` — was 0.5; caching full-res volumes at 0.5 exceeded Colab RAM (`Loading dataset` died at ~30% every time)
- `batch_size: 1` and `samples_per_case: 2` — was `2` and `4`. MONAI's `RandCropByPosNegLabeld` returns `samples_per_case` crops per case, and the DataLoader batches them, so the **effective GPU batch was `2 x 4 = 8`** 128^3 patches, which caused a CUDA OOM (13.27GB / 14.56GB used) on the very first training step
- `patch_size: [96, 96, 96]` — was `[128,128,128]`; smaller patches further reduce activation memory

In [ ]:
import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

config["train"]["epochs"] = 100
config["paths"]["checkpoint_dir"] = "/content/drive/MyDrive/brain_tumor_seg_checkpoints"
config["data"]["cache_rate"] = 0.05
config["train"]["batch_size"] = 1
config["train"]["samples_per_case"] = 2
config["data"]["patch_size"] = [96, 96, 96]

with open("config.yaml", "w") as f:
    yaml.safe_dump(config, f)

print(yaml.safe_dump(config))

## 6. Train
If this still hits a CUDA OOM, rerun after lowering `patch_size` to `[64, 64, 64]` in the cell above (Runtime -> Restart session first to clear pinned GPU memory from the crashed process).

In [ ]:
!python -m src.train --config config.yaml